# FedGNN: Federated Training on Kaggle

Runs the full experiment matrix (local-only, FedAvg, FedProx with prototype alignment, and differentially private variants) across three random seeds (42, 43, 44) on single-GPU Kaggle kernels.

Requires the `<kaggle-username>/fedmagnn-code` Kaggle dataset (staged with `kaggle/stage_kaggle.py`) attached as kernel input.

In [ ]:
import subprocess, sys, os, shutil, glob

def sh(cmd):
    print('$ ' + ' '.join(cmd))
    subprocess.run(cmd, check=True)

# Pascal GPUs (P100, sm_60) are unsupported by Kaggle's stock torch build —
# reinstall a compatible build before the first torch import.
cap = None
try:
    out = subprocess.run(['nvidia-smi','--query-gpu=name,compute_cap','--format=csv,noheader'],
                          capture_output=True, text=True, check=True).stdout.strip().splitlines()[0]
    name, cc = out.split(',')
    cap = tuple(int(x) for x in cc.strip().split('.'))
    print(f'nvidia-smi: {name.strip()} compute_cap={cap}')
except Exception as e:
    print('nvidia-smi check skipped:', e)

if cap is not None and cap < (7, 0):
    print('Reinstalling Pascal-compatible torch (cu121) before first import...')
    sh([sys.executable,'-m','pip','install','-q','--force-reinstall',
        'torch==2.4.1','--index-url','https://download.pytorch.org/whl/cu121'])

import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

sh([sys.executable,'-m','pip','install','-q','torch-geometric','tabulate'])

In [ ]:
# Reduce CUDA fragmentation for the large-graph forwards (inherited by subprocesses)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Locate the attached fedmagnn-code dataset — search nested mount layouts
# (older: /kaggle/input/<slug>; newer: /kaggle/input/datasets/<owner>/<slug>)
candidates = []
for pat in ['/kaggle/input/*', '/kaggle/input/*/*', '/kaggle/input/*/*/*']:
    for d in glob.glob(pat):
        if (os.path.isdir(os.path.join(d, 'presampled'))
                and os.path.exists(os.path.join(d, 'run_federated.py'))):
            candidates.append(d)
if not candidates:
    tree = []
    for root, dirs, files in os.walk('/kaggle/input'):
        depth = root.count('/') - 2
        if depth <= 3:
            tree.append(f"{root} -> dirs={dirs[:8]} files={files[:8]}")
    raise RuntimeError('fedmagnn-code dataset not found under /kaggle/input:\n'
                       + '\n'.join(tree))
SRC = candidates[0]

# Copy code to a writable dir; data stays in read-only input
WORK = '/kaggle/working/FedMAGNN'
if os.path.exists(WORK):
    shutil.rmtree(WORK)
shutil.copytree(SRC, WORK, ignore=shutil.ignore_patterns('presampled'))
os.environ['FEDMAGNN_DATA'] = f'{SRC}/presampled'
os.chdir(WORK)
print('code at', WORK)
print('data  at', os.environ['FEDMAGNN_DATA'])
print(os.listdir(os.environ['FEDMAGNN_DATA']))

In [ ]:
# Zero-shot transfer with calibrated anomaly classifier (Sec. IV-F of the paper).
# ctu13 trains with zero malicious examples. 3 seeds x {align, noalign}.
RESULTS_DIR = '/kaggle/working/results'

EXPERIMENTS = []
for seed in (42, 43, 44):
    EXPERIMENTS += [
        (f'zsc_align_s{seed}',   ['--mode', 'fedavg', '--align', '--zeroshot_silo', 'ctu13', '--seed', str(seed)]),
        (f'zsc_noalign_s{seed}', ['--mode', 'fedavg', '--zeroshot_silo', 'ctu13', '--seed', str(seed)]),
    ]

def run_experiment(tag, extra):
    cmd = [sys.executable, 'run_federated.py',
           '--config', 'configs/fed_default.yaml',
           '--results_dir', RESULTS_DIR,
           '--tag', tag] + extra
    print('\n' + '=' * 70)
    print('RUNNING:', tag)
    print('=' * 70)
    subprocess.run(cmd, check=True)

for tag, extra in EXPERIMENTS:
    run_experiment(tag, extra)

In [ ]:
# Zero-shot summary: naive prototype-NN vs calibrated anomaly threshold
import json
import pandas as pd

rows = []
for tag, _ in EXPERIMENTS:
    path = f'{RESULTS_DIR}/fed_results_{tag}.json'
    if not os.path.exists(path):
        continue
    with open(path) as f:
        r = json.load(f)
    zs = r.get('zeroshot') or {}
    row = {'experiment': tag}
    for kind in ('naive', 'calibrated'):
        m = zs.get(kind) or {}
        row[f'{kind}_f1'] = round(m.get('f1', float('nan')), 2)
        row[f'{kind}_recall'] = round(m.get('recall', float('nan')), 2)
    rows.append(row)

df = pd.DataFrame(rows)
print('ctu13 trained with ZERO malicious examples — detection via foreign DP prototypes')
print(df.to_string(index=False))
df.to_csv(f'{RESULTS_DIR}/summary.csv', index=False)

In [ ]:
# Package results for download
shutil.make_archive('/kaggle/working/fedmagnn_results', 'zip', RESULTS_DIR)
print('saved /kaggle/working/fedmagnn_results.zip')
print(sorted(os.listdir(RESULTS_DIR)))